# ✈️ ELT Pipeline — Flight Delay & Airports Dataset
**FINAL VERSION** — Logging manual (file-flush atomik) sesuai panduan UAS

**Perbedaan ELT vs ETL:**
- Data di-load ke BigQuery **apa adanya** (data lake) terlebih dahulu
- Transformasi dilakukan **di dalam BigQuery** menggunakan SQL
- Tidak ada transformasi Python sebelum load

**Sumber Data:**
- Source 1: `flights_sample_3m.csv` — data keterlambatan penerbangan
- Source 2: `airports.csv` — data bandara global

**Logging:** `write_custom_log()` — open/flush atomik, bukan `logging.basicConfig`

## ⚙️ CELL 1 — Install & Setup

In [ ]:
!pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com -q
!pip install pandas-gbq google-cloud-bigquery db-dtypes -q

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout)
print('✅ Setup selesai')

Sun Jun 14 08:42:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import cudf
import pandas as pd
import numpy as np
import os, time, subprocess
from datetime import datetime
from google.colab import auth
from google.cloud import bigquery
import pandas_gbq

print(f'✅ cuDF    : {cudf.__version__}')
print(f'✅ Pandas  : {pd.__version__}')

gpu_info = subprocess.run(
    ['nvidia-smi','--query-gpu=memory.total,memory.free','--format=csv,noheader'],
    capture_output=True, text=True)
print(f'✅ GPU RAM : {gpu_info.stdout.strip()}')

✅ cuDF    : 26.02.01
✅ Pandas  : 2.2.2
✅ GPU RAM : 15360 MiB, 14913 MiB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/TUBES BIG DATA'
LAKE_DIR = os.path.join(BASE_DIR, 'datalake')
os.makedirs(LAKE_DIR, exist_ok=True)

ROW_LIMIT   = 150_000
GCP_PROJECT = 'tubes-bigdata-498212'   # ← sesuaikan jika perlu
BQ_DATASET  = 'elt_flight_analytics'
BQ_LOCATION = 'US'

print('✅ Folder ELT siap')
print(f'   DATALAKE  → {LAKE_DIR}')
print(f'   Target    → {ROW_LIMIT:,} baris')
print(f'   BigQuery  → {GCP_PROJECT}.{BQ_DATASET}')

Mounted at /content/drive
✅ Folder ELT siap
   DATALAKE  → /content/drive/MyDrive/TUBES BIG DATA/datalake
   Target    → 150,000 baris
   BigQuery  → tubes-bigdata-498212.elt_flight_analytics


## 📝 CELL 4 — Modul Logging Manual (File-Flush Atomik)

In [ ]:
# ============================================================
# LOGGING MANUAL — sesuai panduan UAS Genap 25/26
# Menggantikan logging.basicConfig yang sering 0 bytes di Drive
# Menggunakan open() + f.flush() → sync langsung ke FUSE Drive
# ============================================================

log_path = os.path.join(BASE_DIR, 'elt_log.txt')

def write_custom_log(phase, step, status, message, details=None):
    """
    Menulis log terstruktur secara atomik dan memaksa flush ke Google Drive.

    Args:
        phase   : nama fase pipeline (ELT)
        step    : nama langkah (Extract_Source1, Load_Raw, Transform_SQL, dll.)
        status  : SUCCESS / FAILED / WARNING
        message : pesan ringkas
        details : dict metadata tambahan (opsional)
    """
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log_entry = (
        f'[{timestamp}] [{phase.upper()}] [{step.upper()}] [{status.upper()}]'
        f' - {message}\n'
    )
    if details:
        for k, v in details.items():
            log_entry += f'   > {k}: {v}\n'

    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(log_entry)
        f.flush()   # ← kunci: paksa sync I/O fisik ke Drive

# Tulis header (overwrite di awal setiap run)
with open(log_path, 'w', encoding='utf-8') as f:
    header = (
        '=== PIPELINE BIG DATA ELT LOG - UAS GENAP 25/26 ===\n'
        f'Eksekusi dimulai pada: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
        + '=' * 60 + '\n'
    )
    f.write(header)
    f.flush()

print(f'✅ Log ELT → {log_path}')

✅ Log ELT → /content/drive/MyDrive/TUBES BIG DATA/elt_log.txt


## 🔐 CELL 5 — Autentikasi & Setup BigQuery

In [ ]:
print('Autentikasi Google...')
auth.authenticate_user()
print('✅ Autentikasi berhasil\n')

client = bigquery.Client(project=GCP_PROJECT)

# Buat dataset jika belum ada
dataset_ref          = bigquery.Dataset(f'{GCP_PROJECT}.{BQ_DATASET}')
dataset_ref.location = BQ_LOCATION
client.create_dataset(dataset_ref, exists_ok=True)
print(f'✅ Dataset BigQuery siap : {GCP_PROJECT}.{BQ_DATASET}')

# Helper: upload DataFrame ke BigQuery
def upload_bq(df, table_name, desc=''):
    start_up = time.time()
    df_up = df.copy().reset_index(drop=True)
    df_up = df_up.loc[:, ~df_up.columns.duplicated(keep='first')]
    for col in df_up.columns:
        dtype = str(df_up[col].dtype)
        if dtype == 'category':
            df_up[col] = df_up[col].astype(str).where(
                df_up[col].astype(str) != 'nan', other=None)
        elif 'datetime' in dtype:
            df_up[col] = df_up[col].dt.strftime('%Y-%m-%d').where(
                df_up[col].notna(), other=None)
        elif dtype == 'object':
            df_up[col] = df_up[col].where(df_up[col].notna(), other=None)
    pandas_gbq.to_gbq(
        df_up,
        destination_table=f'{BQ_DATASET}.{table_name}',
        project_id=GCP_PROJECT,
        if_exists='replace',
        progress_bar=True
    )
    dur = round(time.time() - start_up, 2)
    print(f'  ✅ {table_name:30s} → BigQuery ({len(df_up):,} baris | {df_up.shape[1]} kolom | {dur}s) {desc}')
    return len(df_up), dur

print('✅ Helper upload_bq siap')

Autentikasi Google...
✅ Autentikasi berhasil

✅ Dataset BigQuery siap : tubes-bigdata-498212.elt_flight_analytics
✅ Helper upload_bq siap


## 🔵 EXTRACT
### CELL 6 — ELT Extract Sumber 1 & 2

In [ ]:
# ============================================================
# EXTRACT SOURCE 1 — Flight Delay CSV
# Pada ELT: TIDAK ada cleaning/transformasi apapun
# Data disimpan ke datalake apa adanya
# ============================================================

def extract_elt_source1():
    start     = time.time()
    path      = os.path.join(BASE_DIR, 'flights_sample_3m.csv')
    ukuran_mb = round(os.path.getsize(path) / 1024**2, 2)

    print(f'   File asli : {ukuran_mb} MB (3 juta baris)')
    print(f'   Mengambil : {ROW_LIMIT:,} baris')

    # Baca ROW_LIMIT baris — TIDAK ADA CLEANING sama sekali
    df = cudf.read_csv(path, nrows=ROW_LIMIT)

    # Simpan ke datalake APA ADANYA
    lake_path = os.path.join(LAKE_DIR, 'lake_flights.csv')
    df.to_csv(lake_path, index=False)

    duration = round(time.time() - start, 2)

    write_custom_log('ELT', 'Extract_Source1', 'SUCCESS',
                     'Sukses mengekstrak data penerbangan ke datalake.',
                     {
                         'Nama Sumber Data'      : 'flights_sample_3m.csv',
                         'Jumlah Baris Extracted': f'{df.shape[0]:,}',
                         'Jumlah Kolom Extracted': df.shape[1],
                         'Ukuran File Asli'      : f'{ukuran_mb} MB',
                         'Waktu Eksekusi'        : f'{duration} detik',
                         'Path Datalake'         : lake_path,
                         'Catatan'               : 'Data mentah tanpa transformasi'
                     })

    print(f'\n✅ ELT EXTRACT — Sumber 1: Flight Delay & Cancellation')
    print(f'   Jumlah baris  : {df.shape[0]:,}')
    print(f'   Jumlah kolom  : {df.shape[1]}')
    print(f'   Kolom         : {list(df.columns)}')
    print(f'   Waktu eksekusi: {duration} detik')
    print(f'   Disimpan ke   : {lake_path}')
    return df


# ============================================================
# EXTRACT SOURCE 2 — Airports CSV
# ============================================================

def extract_elt_source2():
    start     = time.time()
    path      = os.path.join(BASE_DIR, 'airports.csv')
    ukuran_mb = round(os.path.getsize(path) / 1024**2, 2)

    df = cudf.read_csv(path)

    lake_path = os.path.join(LAKE_DIR, 'lake_airports.csv')
    df.to_csv(lake_path, index=False)

    duration = round(time.time() - start, 2)

    write_custom_log('ELT', 'Extract_Source2', 'SUCCESS',
                     'Sukses mengekstrak data bandara ke datalake.',
                     {
                         'Nama Sumber Data'      : 'airports.csv',
                         'Jumlah Baris Extracted': f'{df.shape[0]:,}',
                         'Jumlah Kolom Extracted': df.shape[1],
                         'Ukuran File Asli'      : f'{ukuran_mb} MB',
                         'Waktu Eksekusi'        : f'{duration} detik',
                         'Path Datalake'         : lake_path,
                         'Catatan'               : 'Data mentah tanpa transformasi'
                     })

    print('\n✅ ELT EXTRACT — Sumber 2: Global Airports')
    print(f'   Jumlah baris  : {df.shape[0]:,}')
    print(f'   Jumlah kolom  : {df.shape[1]}')
    print(f'   Kolom         : {list(df.columns)}')
    print(f'   Ukuran file   : {ukuran_mb} MB')
    print(f'   Waktu eksekusi: {duration} detik')
    print(f'   Disimpan ke   : {lake_path}')
    return df


# ── JALANKAN EXTRACT ──
print('=== EXTRACT PHASE ===')
df_flights  = extract_elt_source1()
df_airports = extract_elt_source2()
print('\n🎉 Semua sumber berhasil diekstrak ke datalake!')

=== EXTRACT PHASE ===
   File asli : 585.69 MB (3 juta baris)
   Mengambil : 150,000 baris

✅ ELT EXTRACT — Sumber 1: Flight Delay & Cancellation
   Jumlah baris  : 150,000
   Jumlah kolom  : 32
   Kolom         : ['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']
   Waktu eksekusi: 3.06 detik
   Disimpan ke   : /content/drive/MyDrive/TUBES BIG DATA/datalake/lake_flights.csv

✅ ELT EXTRACT — Sumber 2: Global Airports
   Jumlah baris  : 6,393
   Jumlah kolom  : 13
   Kolom         : ['AirportName', 'IATA', 'ICAO', 'TimeZone', 'City_Name', 'City_IATA', 'UTC_Offset_Hours', 'UT

### CELL 7 — Observasi Data Mentah

In [ ]:
print('='*55)
print('   INFO DATA MENTAH — df_flights')
print('='*55)
print(f'Shape         : {df_flights.shape}')
print(f'\nTipe data:')
print(df_flights.dtypes.to_string())
print(f'\nMissing value:')
print(df_flights.isnull().sum().to_string())
print(f'\nDuplikat      : {df_flights.duplicated().sum():,}')

print('\n'+'='*55)
print('   INFO DATA MENTAH — df_airports')
print('='*55)
print(f'Shape         : {df_airports.shape}')
print(f'\nTipe data:')
print(df_airports.dtypes.to_string())
print(f'\nMissing value:')
print(df_airports.isnull().sum().to_string())

write_custom_log('ELT', 'Observasi_RawData', 'SUCCESS',
                 'Observasi data mentah selesai.',
                 {
                     'Flights Shape'    : str(df_flights.shape),
                     'Airports Shape'   : str(df_airports.shape),
                     'Flights Duplikat' : f'{df_flights.duplicated().sum():,}',
                     'Airports Missing' : f'{df_airports.isnull().sum().sum():,}',
                     'Catatan'          : 'Data belum ditransformasi, siap LOAD ke BigQuery'
                 })

print('\n✅ Cek data mentah selesai')
print('➡️  Siap LOAD ke BigQuery (data lake)')

   INFO DATA MENTAH — df_flights
Shape         : (150000, 32)

Tipe data:
FL_DATE                     object
AIRLINE                     object
AIRLINE_DOT                 object
AIRLINE_CODE                object
DOT_CODE                     int64
FL_NUMBER                    int64
ORIGIN                      object
ORIGIN_CITY                 object
DEST                        object
DEST_CITY                   object
CRS_DEP_TIME                 int64
DEP_TIME                   float64
DEP_DELAY                  float64
TAXI_OUT                   float64
WHEELS_OFF                 float64
WHEELS_ON                  float64
TAXI_IN                    float64
CRS_ARR_TIME                 int64
ARR_TIME                   float64
ARR_DELAY                  float64
CANCELLED                  float64
CANCELLATION_CODE           object
DIVERTED                   float64
CRS_ELAPSED_TIME           float64
ELAPSED_TIME               float64
AIR_TIME                   float64
DISTANCE        

## 🟢 LOAD (Data Mentah ke BigQuery)
### CELL 8 — Load Raw ke BigQuery Data Lake

In [ ]:
# ============================================================
# ELT LOAD — RAW DATA LANGSUNG KE BIGQUERY
# Tidak ada cleaning, tidak ada transformasi di sini.
# Ini membedakan ELT dari ETL: data masuk warehouse dulu,
# baru ditransformasi di dalam warehouse menggunakan SQL.
# ============================================================

start = time.time()

# cuDF → pandas
df_flights_pd  = df_flights.to_pandas()
df_airports_pd = df_airports.to_pandas()

# Lowercase kolom (hanya standarisasi nama, BUKAN transformasi data)
df_flights_pd.columns  = [c.lower().strip() for c in df_flights_pd.columns]
df_airports_pd.columns = [c.lower().strip() for c in df_airports_pd.columns]

print('Upload raw data ke BigQuery (data lake)...\n')

n_flights,  dur_flights  = upload_bq(df_flights_pd,  'raw_flights',  '[data lake — mentah]')
n_airports, dur_airports = upload_bq(df_airports_pd, 'raw_airports', '[data lake — mentah]')

duration = round(time.time() - start, 2)

write_custom_log('ELT', 'Load_Raw_BigQuery', 'SUCCESS',
                 'Data mentah berhasil dimuat ke BigQuery sebagai data lake.',
                 {
                     'raw_flights Baris'   : f'{n_flights:,}',
                     'raw_flights Load Time': f'{dur_flights} detik',
                     'raw_airports Baris'  : f'{n_airports:,}',
                     'raw_airports Load Time': f'{dur_airports} detik',
                     'Target Dataset'      : f'{GCP_PROJECT}.{BQ_DATASET}',
                     'Total Waktu Load'    : f'{duration} detik',
                     'Catatan'             : 'ELT Load: data masuk warehouse tanpa transformasi'
                 })

print(f'\n   Waktu load total : {duration} detik')
print('✅ ELT Load selesai — data mentah tersimpan di BigQuery sebagai data lake')

Upload raw data ke BigQuery (data lake)...



100%|██████████| 1/1 [00:00<00:00, 8905.10it/s]


  ✅ raw_flights                    → BigQuery (150,000 baris | 32 kolom | 10.95s) [data lake — mentah]


100%|██████████| 1/1 [00:00<00:00, 11650.84it/s]

  ✅ raw_airports                   → BigQuery (6,393 baris | 13 kolom | 3.4s) [data lake — mentah]

   Waktu load total : 14.55 detik
✅ ELT Load selesai — data mentah tersimpan di BigQuery sebagai data lake


## 🟡 TRANSFORM (SQL di BigQuery)
### CELL 9 — Transform Step 1: Cleaning via SQL

In [ ]:
# ============================================================
# TRANSFORM STEP 1 — CLEANING VIA BIGQUERY SQL
# Seluruh transformasi dilakukan DI DALAM BigQuery.
# Inilah perbedaan utama ELT vs ETL.
#
# FIX #1 : COALESCE(dep_delay, 0) dihapus untuk cancelled flights
#           → cancelled dibiarkan NULL di dep_delay
# FIX #2 : Range check pakai BETWEEN -120 AND 1440 (hard cap)
# FIX #3 : delay_status beri kategori 'cancelled' tersendiri
# FIX #4 : INNER JOIN → LEFT JOIN agar data tidak hilang
# ============================================================

print('TRANSFORM Step 1 — Cleaning via BigQuery SQL...\n')
start_time = time.time()

# ── Cleaning flights ─────────────────────────────────────────
sql_clean_flights = f"""
CREATE OR REPLACE TABLE `{GCP_PROJECT}.{BQ_DATASET}.elt_cleaned_flights` AS
WITH deduped AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY fl_date, airline, origin, dest
               ORDER BY dep_time ASC NULLS LAST
           ) AS rn
    FROM `{GCP_PROJECT}.{BQ_DATASET}.raw_flights`
    WHERE fl_date  IS NOT NULL
      AND airline  IS NOT NULL
      AND origin   IS NOT NULL
      AND dest     IS NOT NULL
)
SELECT
    PARSE_DATE('%Y-%m-%d', fl_date)            AS fl_date,
    airline,
    origin,
    dest,
    dep_time,
    -- FIX #1: NULL dep_delay untuk cancelled dibiarkan NULL
    CASE
        WHEN COALESCE(cancelled, 0) = 1 THEN NULL
        ELSE COALESCE(CAST(dep_delay AS FLOAT64), 0.0)
    END                                        AS dep_delay,
    CASE
        WHEN COALESCE(cancelled, 0) = 1 THEN NULL
        ELSE COALESCE(CAST(arr_delay AS FLOAT64), 0.0)
    END                                        AS arr_delay,
    COALESCE(CAST(air_time  AS FLOAT64), 0.0)  AS air_time,
    COALESCE(CAST(distance  AS FLOAT64), 0.0)  AS distance,
    COALESCE(CAST(cancelled AS INT64),   0)    AS cancelled
FROM deduped
WHERE rn = 1
  -- FIX #2: hard cap range -120 s.d. 1440 menit (hanya non-cancelled)
  AND (
      COALESCE(cancelled, 0) = 1
      OR CAST(dep_delay AS FLOAT64) BETWEEN -120 AND 1440
  )
"""

# ── Cleaning airports ────────────────────────────────────────
sql_clean_airports = f"""
CREATE OR REPLACE TABLE `{GCP_PROJECT}.{BQ_DATASET}.elt_cleaned_airports` AS
SELECT
    iata                               AS iata_code,
    airportname                        AS airport_name,
    city_name                          AS city,
    country_name                       AS country,
    SAFE_CAST(geopointlat  AS FLOAT64) AS latitude,
    SAFE_CAST(geopointlong AS FLOAT64) AS longitude,
    timezone
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY iata ORDER BY iata) AS rn
    FROM `{GCP_PROJECT}.{BQ_DATASET}.raw_airports`
    WHERE iata IS NOT NULL AND TRIM(iata) != ''
)
WHERE rn = 1
"""

for nama, sql in [('elt_cleaned_flights', sql_clean_flights),
                  ('elt_cleaned_airports', sql_clean_airports)]:
    job = client.query(sql)
    job.result()
    tbl = client.get_table(f'{GCP_PROJECT}.{BQ_DATASET}.{nama}')
    print(f'  ✅ {nama:25s} : {tbl.num_rows:,} baris | {len(tbl.schema)} kolom')

duration = round(time.time() - start_time, 2)

write_custom_log('ELT', 'Transform_SQL_Cleaning', 'SUCCESS',
                 'Cleaning via BigQuery SQL selesai.',
                 {
                     'Metode'                   : 'SQL CREATE OR REPLACE TABLE di BigQuery',
                     'Tabel Dibuat'             : 'elt_cleaned_flights, elt_cleaned_airports',
                     'FIX Diterapkan'           : 'NULL cancelled, hard cap [-120,1440], dedup ROW_NUMBER',
                     'Waktu Eksekusi SQL'       : f'{duration} detik'
                 })

print('\n✅ Cleaning selesai')

TRANSFORM Step 1 — Cleaning via BigQuery SQL...

  ✅ elt_cleaned_flights       : 148,843 baris | 10 kolom
  ✅ elt_cleaned_airports      : 6,372 baris | 7 kolom

✅ Cleaning selesai


### CELL 10 — Transform Step 2: Merge & Feature Engineering via SQL

In [ ]:
# ============================================================
# TRANSFORM STEP 2 — MERGE & FEATURE ENGINEERING
# FIX #4 : INNER JOIN origin → LEFT JOIN
# FIX #5 : delay_status beri label 'cancelled' tersendiri
# FIX #6 : tambah fitur quarter sebagai fitur ke-6
# ============================================================

print('TRANSFORM Step 2 — Merge & Feature Engineering via BigQuery SQL...\n')
start_time = time.time()

sql_transform = f"""
CREATE OR REPLACE TABLE `{GCP_PROJECT}.{BQ_DATASET}.elt_transformed` AS
SELECT
    f.fl_date,
    f.airline,
    f.origin,
    f.dest,
    f.dep_delay,
    f.arr_delay,
    f.air_time,
    f.distance,
    f.cancelled,

    -- Info bandara ASAL
    ao.airport_name  AS origin_airport_name,
    ao.city          AS origin_city,
    ao.country       AS origin_country,
    ao.latitude      AS origin_lat,
    ao.longitude     AS origin_lon,
    ao.timezone      AS origin_timezone,

    -- Info bandara TUJUAN
    ad.airport_name  AS dest_airport_name,
    ad.city          AS dest_city,
    ad.country       AS dest_country,
    ad.latitude      AS dest_lat,
    ad.longitude     AS dest_lon,

    -- Fitur 1: Tahun penerbangan
    EXTRACT(YEAR  FROM f.fl_date)                  AS flight_year,

    -- Fitur 2: Bulan penerbangan
    EXTRACT(MONTH FROM f.fl_date)                  AS flight_month,

    -- Fitur 3: Hari dalam seminggu (0=Minggu, 6=Sabtu)
    EXTRACT(DAYOFWEEK FROM f.fl_date) - 1          AS day_of_week,

    -- Fitur 4: Status delay
    -- FIX #5: cancelled diberi label tersendiri, tidak masuk on_time
    CASE
        WHEN f.cancelled = 1       THEN 'cancelled'
        WHEN f.dep_delay IS NULL   THEN 'unknown'
        WHEN f.dep_delay <= 0      THEN 'on_time'
        WHEN f.dep_delay <= 15     THEN 'minor_delay'
        WHEN f.dep_delay <= 60     THEN 'moderate_delay'
        ELSE                            'severe_delay'
    END                                            AS delay_status,

    -- Fitur 5: Is Weekend
    CASE
        WHEN EXTRACT(DAYOFWEEK FROM f.fl_date) IN (1, 7) THEN 1
        ELSE 0
    END                                            AS is_weekend,

    -- Fitur 6: Quarter (FIX #6: fitur tambahan)
    EXTRACT(QUARTER FROM f.fl_date)                AS quarter

FROM `{GCP_PROJECT}.{BQ_DATASET}.elt_cleaned_flights` f
-- FIX #4: LEFT JOIN agar penerbangan tidak hilang jika bandara tidak ada
LEFT JOIN `{GCP_PROJECT}.{BQ_DATASET}.elt_cleaned_airports` ao ON f.origin = ao.iata_code
LEFT JOIN `{GCP_PROJECT}.{BQ_DATASET}.elt_cleaned_airports` ad ON f.dest   = ad.iata_code
"""

job = client.query(sql_transform)
job.result()

tbl = client.get_table(f'{GCP_PROJECT}.{BQ_DATASET}.elt_transformed')
print(f'  ✅ elt_transformed : {tbl.num_rows:,} baris | {len(tbl.schema)} kolom')

# Cek null origin_airport_name (info saja — bukan error fatal)
q = client.query(f"""
    SELECT COUNT(*) AS null_origin
    FROM `{GCP_PROJECT}.{BQ_DATASET}.elt_transformed`
    WHERE origin_airport_name IS NULL
""")
null_origin = list(q.result())[0]['null_origin']
print(f'  origin_airport_name NULL : {null_origin:,} baris (bandara tidak ada di airports.csv)')

duration = round(time.time() - start_time, 2)

write_custom_log('ELT', 'Transform_SQL_MergeFE', 'SUCCESS',
                 'Merge & Feature Engineering via BigQuery SQL selesai.',
                 {
                     'Tabel Dibuat'     : 'elt_transformed',
                     'Baris Hasil'      : f'{tbl.num_rows:,}',
                     'Kolom Hasil'      : len(tbl.schema),
                     'Fitur Baru'       : 'flight_year, flight_month, day_of_week, delay_status, is_weekend, quarter',
                     'JOIN Strategy'    : 'LEFT JOIN (origin & dest) ke elt_cleaned_airports',
                     'NULL Origin Name' : f'{null_origin:,} baris',
                     'FIX Diterapkan'   : 'LEFT JOIN, delay_status cancelled label, fitur quarter',
                     'Waktu Eksekusi SQL': f'{duration} detik'
                 })

print('\n✅ Merge & Feature Engineering selesai')

TRANSFORM Step 2 — Merge & Feature Engineering via BigQuery SQL...

  ✅ elt_transformed : 148,843 baris | 26 kolom
  origin_airport_name NULL : 176 baris (bandara tidak ada di airports.csv)

✅ Merge & Feature Engineering selesai


### CELL 11 — Transform Step 3: Agregasi & Metrik via SQL

In [ ]:
# ============================================================
# TRANSFORM STEP 3 — AGREGASI METRIK
# FIX #7 : avg_dep_delay dan avg_arr_delay exclude cancelled
# FIX #8 : tambah on_time_rate per maskapai per bulan
# ============================================================

print('TRANSFORM Step 3 — Agregasi Metrik via BigQuery SQL...\n')
start_time = time.time()

sql_metrics = f"""
CREATE OR REPLACE TABLE `{GCP_PROJECT}.{BQ_DATASET}.elt_metrics` AS
SELECT
    airline,
    flight_year,
    flight_month,
    origin_country,

    COUNT(*)                                                   AS total_flights,

    -- FIX #7: avg delay hanya untuk yang tidak cancelled
    ROUND(AVG(CASE WHEN cancelled = 0 THEN dep_delay END), 2)  AS avg_dep_delay,
    ROUND(AVG(CASE WHEN cancelled = 0 THEN arr_delay END), 2)  AS avg_arr_delay,

    ROUND(AVG(distance), 1)                                    AS avg_distance,
    SUM(cancelled)                                             AS total_cancelled,
    ROUND(AVG(CASE WHEN cancelled = 0 THEN air_time END), 1)   AS avg_air_time,

    -- FIX #8: on_time_rate yang benar (exclude cancelled dari denominator)
    ROUND(
        SAFE_DIVIDE(
            COUNTIF(delay_status = 'on_time'),
            NULLIF(COUNTIF(cancelled = 0), 0)
        ) * 100, 2
    )                                                          AS on_time_rate_pct

FROM `{GCP_PROJECT}.{BQ_DATASET}.elt_transformed`
WHERE flight_year IS NOT NULL
GROUP BY airline, flight_year, flight_month, origin_country
ORDER BY flight_year, flight_month, total_flights DESC
"""

job = client.query(sql_metrics)
job.result()

tbl = client.get_table(f'{GCP_PROJECT}.{BQ_DATASET}.elt_metrics')
print(f'  ✅ elt_metrics : {tbl.num_rows:,} baris')

# Preview
df_preview = client.query(f"""
    SELECT * FROM `{GCP_PROJECT}.{BQ_DATASET}.elt_metrics`
    ORDER BY flight_year, flight_month
    LIMIT 10
""").to_dataframe()
print(df_preview.to_string(index=False))

duration = round(time.time() - start_time, 2)

write_custom_log('ELT', 'Transform_SQL_Metrics', 'SUCCESS',
                 'Agregasi metrik via BigQuery SQL selesai.',
                 {
                     'Tabel Dibuat'     : 'elt_metrics',
                     'Baris Hasil'      : f'{tbl.num_rows:,}',
                     'Metrik Dihitung'  : 'avg_dep_delay, avg_arr_delay, avg_distance, total_cancelled, avg_air_time, on_time_rate_pct',
                     'FIX Diterapkan'   : 'AVG exclude cancelled, on_time_rate benar',
                     'Waktu Eksekusi SQL': f'{duration} detik'
                 })

print('\n✅ Agregasi metrik selesai')

TRANSFORM Step 3 — Agregasi Metrik via BigQuery SQL...

  ✅ elt_metrics : 1,492 baris
               airline  flight_year  flight_month           origin_country  total_flights  avg_dep_delay  avg_arr_delay  avg_distance  total_cancelled  avg_air_time  on_time_rate_pct
             Envoy Air         2019             1 United States of America            145          13.17          12.18         437.4               11          72.5             62.69
    Mesa Airlines Inc.         2019             1 United States of America            107           9.45           8.03         619.4                3          91.6             75.96
American Airlines Inc.         2019             1 United States of America            383           6.30           1.55        1024.5                8         138.3             66.40
      Republic Airline         2019             1 United States of America            125          10.37           7.83         603.5                5          90.9             68.33

### CELL 12 — Validasi Kualitas Data (6 aturan via SQL)

In [ ]:
# ============================================================
# VALIDASI 6 ATURAN — semua via BigQuery SQL
# FIX #9 : distribusi cek mean dep_delay harus > 0
# FIX #10: on-time rate dihitung dengan formula benar
# ============================================================

print('='*55)
print('   VALIDASI KUALITAS DATA ELT')
print('='*55)
start_time = time.time()

hasil = {}

def bq(sql):
    return list(client.query(sql).result())[0][0]

BASE_TBL  = f'`{GCP_PROJECT}.{BQ_DATASET}.elt_cleaned_flights`'
TRANS_TBL = f'`{GCP_PROJECT}.{BQ_DATASET}.elt_transformed`'

# 1. Uniqueness
dup = bq(f"""
    SELECT COUNT(*) - COUNT(DISTINCT CONCAT(CAST(fl_date AS STRING), airline, origin, dest))
    FROM {BASE_TBL}
""")
hasil['1. Uniqueness'] = 'PASS ✅' if dup == 0 else f'FAIL ❌ ({dup} duplikat)'

# 2. Null check kolom kritis
nulls = bq(f"""
    SELECT COUNT(*) FROM {BASE_TBL}
    WHERE airline IS NULL OR origin IS NULL OR dest IS NULL
""")
hasil['2. Null Check'] = 'PASS ✅' if nulls == 0 else f'FAIL ❌ ({nulls} nulls)'

# 3. Range check dep_delay (hanya non-cancelled)
out_range = bq(f"""
    SELECT COUNT(*) FROM {BASE_TBL}
    WHERE cancelled = 0
      AND (dep_delay < -120 OR dep_delay > 1440)
""")
hasil['3. Range Check'] = 'PASS ✅' if out_range == 0 else f'FAIL ❌ ({out_range} nilai)'

# 4. Datatype date
dtype_rows = list(client.query(f"""
    SELECT data_type FROM `{GCP_PROJECT}.{BQ_DATASET}`.INFORMATION_SCHEMA.COLUMNS
    WHERE table_name = 'elt_cleaned_flights' AND column_name = 'fl_date'
""").result())
dtype = dtype_rows[0][0] if dtype_rows else 'NOT FOUND'
hasil['4. Datatype Date'] = f'PASS ✅ ({dtype})' if 'DATE' in dtype.upper() else f'FAIL ❌ ({dtype})'

# 5. Referential integrity
no_match = bq(f"SELECT COUNT(*) FROM {TRANS_TBL} WHERE origin_city IS NULL")
total    = bq(f"SELECT COUNT(*) FROM {TRANS_TBL}")
match_pct = round((total - no_match) / total * 100, 1) if total > 0 else 0
hasil['5. Referential Integrity'] = (
    'PASS ✅ (100% match)' if no_match == 0
    else f'WARN ⚠️ ({no_match:,} origin tidak match | match rate: {match_pct}%)')

# 6. Distribusi dep_delay — FIX #9: mean harus > 0 dan std > 5
stats = list(client.query(f"""
    SELECT
        ROUND(AVG(dep_delay), 2)    AS mean_delay,
        ROUND(STDDEV(dep_delay), 2) AS std_delay
    FROM {BASE_TBL}
    WHERE cancelled = 0 AND dep_delay IS NOT NULL
""").result())[0]
mean_d, std_d = stats['mean_delay'], stats['std_delay']
dist_ok = (std_d is not None and std_d > 5) and (mean_d is not None and -10 < mean_d < 60)
hasil['6. Distribusi'] = (
    f'PASS ✅ (mean={mean_d} mnt, std={std_d})'
    if dist_ok
    else f'WARN ⚠️ (mean={mean_d} mnt, std={std_d}) — distribusi mencurigakan')

# Print hasil
for k, v in hasil.items():
    print(f'  {k}: {v}')

# FIX #10: On-Time Rate verifikasi dengan formula benar
otr = bq(f"""
    SELECT ROUND(
        SAFE_DIVIDE(COUNTIF(delay_status = 'on_time'),
                    NULLIF(COUNTIF(cancelled = 0), 0)) * 100, 2)
    FROM {TRANS_TBL}
""")

duration = round(time.time() - start_time, 2)

print('='*55)
print(f'\n  Total baris (transformed) : {total:,}')
print(f'  On-Time Rate (verifikasi) : {otr}%')
print('='*55)

write_custom_log('ELT', 'Validasi_KualitasData', 'SUCCESS',
                 'Validasi 6 aturan kualitas data via BigQuery SQL selesai.',
                 {
                     'Uniqueness Check'     : hasil['1. Uniqueness'],
                     'Null Check Kritis'    : hasil['2. Null Check'],
                     'Range Check Delay'    : hasil['3. Range Check'],
                     'Datatype Date'        : hasil['4. Datatype Date'],
                     'Referential Integrity': hasil['5. Referential Integrity'],
                     'Distribusi Data'      : hasil['6. Distribusi'],
                     'On-Time Rate'         : f'{otr}%',
                     'Total Baris'          : f'{total:,}',
                     'Waktu Validasi SQL'   : f'{duration} detik'
                 })

print('\n✅ Validasi kualitas data ELT selesai!')

   VALIDASI KUALITAS DATA ELT
  1. Uniqueness: PASS ✅
  2. Null Check: PASS ✅
  3. Range Check: PASS ✅
  4. Datatype Date: PASS ✅ (DATE)
  5. Referential Integrity: WARN ⚠️ (176 origin tidak match | match rate: 99.9%)
  6. Distribusi: PASS ✅ (mean=10.09 mnt, std=48.61)

  Total baris (transformed) : 148,843
  On-Time Rate (verifikasi) : 66.04%

✅ Validasi kualitas data ELT selesai!


## ✅ CELL 13 — Summary Tabel BigQuery & Verifikasi Log

In [ ]:
print('\n' + '='*60)
print('   SUMMARY TABEL ELT DI BIGQUERY')
print('='*60)

tables = [
    ('raw_flights',          'Data lake — mentah'),
    ('raw_airports',         'Data lake — mentah'),
    ('elt_cleaned_flights',  'Hasil cleaning SQL'),
    ('elt_cleaned_airports', 'Hasil cleaning SQL'),
    ('elt_transformed',      'Hasil merge + feature engineering SQL'),
    ('elt_metrics',          'Hasil agregasi metrik SQL'),
]

total_all = 0
for tbl_name, keterangan in tables:
    try:
        tbl = client.get_table(f'{GCP_PROJECT}.{BQ_DATASET}.{tbl_name}')
        total_all += tbl.num_rows
        print(f'  {tbl_name:30s} : {tbl.num_rows:>9,} baris | {len(tbl.schema):2} kolom  ({keterangan})')
    except Exception as e:
        print(f'  {tbl_name:30s} : ERROR — {e}')

print('='*60)
print(f'  Total baris keseluruhan  : {total_all:,}')
print(f'  Dataset                  : {GCP_PROJECT}.{BQ_DATASET}')
print('='*60)

# ── VERIFIKASI LOG ────────────────────────────────────────────
print('\n=== VERIFIKASI LOG ===')
if os.path.exists(log_path):
    size_kb = round(os.path.getsize(log_path) / 1024, 2)
    print(f'✅ elt_log.txt ditemukan: {size_kb} KB')
    print(f'   Path: {log_path}\n')
    print('--- ISI LOG (50 baris terakhir) ---')
    with open(log_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(''.join(lines[-50:]))
else:
    print('❌ elt_log.txt TIDAK ditemukan — cek BASE_DIR!')

# Tulis ringkasan akhir ke log
write_custom_log('ELT', 'Pipeline_Summary', 'SUCCESS',
                 'Seluruh pipeline ELT selesai dijalankan.',
                 {
                     'Total Tabel BigQuery' : len(tables),
                     'Total Baris'          : f'{total_all:,}',
                     'Dataset'              : f'{GCP_PROJECT}.{BQ_DATASET}',
                     'Datalake'             : LAKE_DIR,
                     'Strategi Transform'   : 'SQL di BigQuery (bukan Python pra-load)',
                 })

# Tulis footer
with open(log_path, 'a', encoding='utf-8') as f:
    footer = (
        '\n' + '=' * 60 + '\n'
        f'Pipeline selesai pada: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
        '=== END OF ELT LOG ===\n'
    )
    f.write(footer)
    f.flush()

print('\n🎉 PIPELINE ELT SELESAI SEPENUHNYA!')
print(f'   Log      → {log_path}')
print(f'   Datalake → {LAKE_DIR}')
print(f'   BigQuery → {GCP_PROJECT}.{BQ_DATASET}')


   SUMMARY TABEL ELT DI BIGQUERY
  raw_flights                    :   150,000 baris | 32 kolom  (Data lake — mentah)
  raw_airports                   :     6,393 baris | 13 kolom  (Data lake — mentah)
  elt_cleaned_flights            :   148,843 baris | 10 kolom  (Hasil cleaning SQL)
  elt_cleaned_airports           :     6,372 baris |  7 kolom  (Hasil cleaning SQL)
  elt_transformed                :   148,843 baris | 26 kolom  (Hasil merge + feature engineering SQL)
  elt_metrics                    :     1,492 baris | 11 kolom  (Hasil agregasi metrik SQL)
  Total baris keseluruhan  : 461,943
  Dataset                  : tubes-bigdata-498212.elt_flight_analytics

=== VERIFIKASI LOG ===
✅ elt_log.txt ditemukan: 3.28 KB
   Path: /content/drive/MyDrive/TUBES BIG DATA/elt_log.txt

--- ISI LOG (50 baris terakhir) ---
   > Jumlah Baris Extracted: 6,393
   > Jumlah Kolom Extracted: 13
   > Ukuran File Asli: 0.65 MB
   > Waktu Eksekusi: 1.07 detik
   > Path Datalake: /content/drive/MyDrive/TU